# NCEI Raw Data Quality

**Purpose.** Audit the files ingested for this provider and the corresponding `raw.*`
DuckDB tables before any normalization, blending, or analytical transformation.

This notebook covers the supplied files/tables, observation grain, date and geography
coverage, column types and meanings, missingness and suppression, duplicate/invalid
keys, numeric ranges, suspicious values, source limitations, and downstream readiness.

## Setup and provider rules

In [1]:
from pathlib import Path
import re
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

ROOT = Path.cwd()
while not (ROOT / "data" / "quoll.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DB_PATH = ROOT / "data" / "quoll.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

PROVIDER = 'ncei'
TABLE_PATTERNS = ['ncei_%']
PRIMARY_PATTERNS = ['ncei_climate_at_a_glance_county_monthly']
KEY_CANDIDATES = [['fips', 'date', 'parameter']]
DATE_CANDIDATES = ['date', 'year', 'month', 'fetched_at']
GEO_CANDIDATES = ['fips', 'state', 'county']
NUMERIC_HINTS = ['year', 'month', 'value', 'anomaly', 'rank']
SUPPRESSION_CODES = ['', 'null']

def matches(name, patterns):
    return any(re.fullmatch(pattern.replace("%", ".*"), name, flags=re.I) for pattern in patterns)

def qi(value):
    return '"' + value.replace('"', '""') + '"'

raw_tables = con.execute(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_schema = 'raw' ORDER BY table_name"
).df()["table_name"].tolist()
provider_tables = [name for name in raw_tables if matches(name, TABLE_PATTERNS)]
primary_tables = [name for name in provider_tables if matches(name, PRIMARY_PATTERNS)]
provider_tables, primary_tables

(['ncei_climate_at_a_glance_county_monthly'],
 ['ncei_climate_at_a_glance_county_monthly'])

## Files and tables supplied

In [2]:
file_inventory = con.execute(
    '''
    SELECT table_name, filename, source_folder, source_path,
           loaded_at, row_count, detected_columns,
           upstream_source_url, content_sha256
    FROM meta.files
    WHERE table_schema = 'raw'
    ORDER BY table_name
    '''
).df()
file_inventory = file_inventory.loc[file_inventory["table_name"].isin(provider_tables)]

table_rows = []
for table in provider_tables:
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    column_count = con.execute(
        "SELECT count(*) FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).fetchone()[0]
    table_rows.append({"table_name": table, "rows": row_count, "columns": column_count,
                       "primary_data_table": table in primary_tables})
table_inventory = pd.DataFrame(table_rows)
display(file_inventory)
display(table_inventory)

,table_name,filename,source_folder,source_path,loaded_at,row_count,detected_columns,upstream_source_url,content_sha256
51,ncei_climate_at_a_glance_county_monthly,ncei_climate_at_a_glance_county_monthly.csv,climate,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:55:43.948845+00:00,769440,"[""fips"", ""parameter"", ""parameter_label"", ""unit...",https://www.ncei.noaa.gov/access/monitoring/cl...,311d5787e8e814f37fa67c8149b43a3edb109c9b9b5985...


,table_name,rows,columns,primary_data_table
0,ncei_climate_at_a_glance_county_monthly,769440,13,True


## Observation grain

One county-month-parameter observation from Climate at a Glance.

The checks below infer candidate keys from the raw columns. A repeated candidate key is
reported rather than silently removed because some provider tables legitimately contain
additional dimensions.

## Column types and meanings

In [3]:
schema_frames = []
for table in primary_tables:
    schema = con.execute(f"DESCRIBE raw.{qi(table)}").df()
    schema.insert(0, "table_name", table)
    schema["inferred_meaning"] = (
        schema["column_name"].str.replace("_", " ", regex=False)
        .str.replace(r"(?<=[a-z])(?=[A-Z])", " ", regex=True)
        .str.strip()
    )
    schema_frames.append(schema)
schema_inventory = pd.concat(schema_frames, ignore_index=True) if schema_frames else pd.DataFrame()
display(schema_inventory)

,table_name,column_name,column_type,null,key,default,extra,inferred_meaning
0,ncei_climate_at_a_glance_county_monthly,fips,VARCHAR,YES,None,None,None,fips
1,ncei_climate_at_a_glance_county_monthly,parameter,VARCHAR,YES,None,None,None,parameter
2,ncei_climate_at_a_glance_county_monthly,parameter_label,VARCHAR,YES,None,None,None,parameter label
3,ncei_climate_at_a_glance_county_monthly,unit,VARCHAR,YES,None,None,None,unit
4,ncei_climate_at_a_glance_county_monthly,year_month,VARCHAR,YES,None,None,None,year month
5,ncei_climate_at_a_glance_county_monthly,date,VARCHAR,YES,None,None,None,date
6,ncei_climate_at_a_glance_county_monthly,year,VARCHAR,YES,None,None,None,year
7,ncei_climate_at_a_glance_county_monthly,month,VARCHAR,YES,None,None,None,month
8,ncei_climate_at_a_glance_county_monthly,value,VARCHAR,YES,None,None,None,value
9,ncei_climate_at_a_glance_county_monthly,anomaly,VARCHAR,YES,None,None,None,anomaly


## Date and geographic coverage

In [4]:
coverage_rows = []
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"].tolist()
    row = {"table_name": table}
    for column in DATE_CANDIDATES:
        if column in columns:
            normalized_column = column.lower()
            if normalized_column == "year" or normalized_column.endswith("_year"):
                coverage_type = "INTEGER"
            elif normalized_column == "month" or normalized_column.endswith("_month"):
                coverage_type = "INTEGER"
            else:
                coverage_type = "TIMESTAMP"
            values = con.execute(
                f"SELECT min(try_cast({qi(column)} AS {coverage_type})), "
                f"max(try_cast({qi(column)} AS {coverage_type})) "
                f"FROM raw.{qi(table)}"
            ).fetchone()
            row[f"{column}_min"] = values[0]
            row[f"{column}_max"] = values[1]
    for column in GEO_CANDIDATES:
        if column in columns:
            row[f"{column}_distinct"] = con.execute(
                f"SELECT count(DISTINCT {qi(column)}) FROM raw.{qi(table)}"
            ).fetchone()[0]
    coverage_rows.append(row)
coverage = pd.DataFrame(coverage_rows)
display(coverage)

,table_name,date_min,date_max,year_min,year_max,month_min,month_max,fetched_at_min,fetched_at_max,fips_distinct
0,ncei_climate_at_a_glance_county_monthly,2016-01-01,2025-12-01,2016,2025,1,12,2026-06-30 02:19:46.223200,2026-06-30 02:26:15.644247,1603


## Missingness and suppression codes

In [5]:
missing_rows = []
suppression_rows = []
suppression_sql = ", ".join("?" for _ in SUPPRESSION_CODES)
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=? ORDER BY ordinal_position", [table]
    ).df()["column_name"].tolist()
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    # Profile all columns for compact tables and the first 80 for unusually wide sources.
    for column in columns[:80]:
        null_count, blank_count = con.execute(
            f"SELECT count(*) FILTER (WHERE {qi(column)} IS NULL), "
            f"count(*) FILTER (WHERE trim(cast({qi(column)} AS VARCHAR))='') "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        missing_rows.append({
            "table_name": table, "column_name": column,
            "missing_count": null_count + blank_count,
            "missing_pct": (null_count + blank_count) / row_count * 100 if row_count else np.nan,
        })
        if SUPPRESSION_CODES:
            suppressed = con.execute(
                f"SELECT count(*) FROM raw.{qi(table)} "
                f"WHERE trim(cast({qi(column)} AS VARCHAR)) IN ({suppression_sql})",
                SUPPRESSION_CODES,
            ).fetchone()[0]
            if suppressed:
                suppression_rows.append({
                    "table_name": table, "column_name": column,
                    "suppression_or_sentinel_count": suppressed,
                })
missingness = pd.DataFrame(missing_rows).sort_values(
    ["missing_pct", "table_name"], ascending=[False, True]
)
suppression = pd.DataFrame(suppression_rows)
display(missingness)
display(suppression if not suppression.empty else pd.DataFrame(
    {"result": ["No configured literal suppression codes were present in profiled columns; nulls remain material."]}
))

,table_name,column_name,missing_count,missing_pct
9,ncei_climate_at_a_glance_county_monthly,anomaly,769440,100.0
10,ncei_climate_at_a_glance_county_monthly,rank,769440,100.0
0,ncei_climate_at_a_glance_county_monthly,fips,0,0.0
1,ncei_climate_at_a_glance_county_monthly,parameter,0,0.0
2,ncei_climate_at_a_glance_county_monthly,parameter_label,0,0.0
3,ncei_climate_at_a_glance_county_monthly,unit,0,0.0
4,ncei_climate_at_a_glance_county_monthly,year_month,0,0.0
5,ncei_climate_at_a_glance_county_monthly,date,0,0.0
6,ncei_climate_at_a_glance_county_monthly,year,0,0.0
7,ncei_climate_at_a_glance_county_monthly,month,0,0.0


,result
0,No configured literal suppression codes were p...


## Duplicate or invalid keys

In [6]:
key_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    keys = next((candidate for candidate in KEY_CANDIDATES if set(candidate).issubset(columns)), [])
    if not keys:
        key_rows.append({"table_name": table, "candidate_key": None,
                         "duplicate_key_groups": np.nan, "invalid_key_rows": np.nan})
        continue
    key_expr = ", ".join(qi(column) for column in keys)
    invalid = " OR ".join(
        f"{qi(column)} IS NULL OR trim(cast({qi(column)} AS VARCHAR))=''" for column in keys
    )
    duplicate_groups = con.execute(
        f"SELECT count(*) FROM (SELECT {key_expr}, count(*) n "
        f"FROM raw.{qi(table)} GROUP BY {key_expr} HAVING count(*) > 1)"
    ).fetchone()[0]
    invalid_rows = con.execute(
        f"SELECT count(*) FROM raw.{qi(table)} WHERE {invalid}"
    ).fetchone()[0]
    key_rows.append({"table_name": table, "candidate_key": " + ".join(keys),
                     "duplicate_key_groups": duplicate_groups,
                     "invalid_key_rows": invalid_rows})
key_quality = pd.DataFrame(key_rows)
display(key_quality)

,table_name,candidate_key,duplicate_key_groups,invalid_key_rows
0,ncei_climate_at_a_glance_county_monthly,fips + date + parameter,0,0


## Numeric ranges and suspicious values

In [7]:
numeric_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    for column in [name for name in NUMERIC_HINTS if name in columns]:
        numeric = (
            f"try_cast(replace(trim(cast({qi(column)} AS VARCHAR)), ',', '') AS DOUBLE)"
        )
        result = con.execute(
            f"SELECT count(*) FILTER (WHERE {numeric} IS NOT NULL), "
            f"min({numeric}), max({numeric}), "
            f"count(*) FILTER (WHERE {numeric} < 0) "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        numeric_rows.append({
            "table_name": table, "column_name": column,
            "numeric_count": result[0], "minimum": result[1],
            "maximum": result[2], "negative_count": result[3],
            "review_flag": (
                "review negative values/sentinels" if result[3] else
                "review extreme min/max against provider definition"
            ),
        })
numeric_ranges = pd.DataFrame(numeric_rows)
display(numeric_ranges)

,table_name,column_name,numeric_count,minimum,maximum,negative_count,review_flag
0,ncei_climate_at_a_glance_county_monthly,year,769440,2016.0,2025.0,0,review extreme min/max against provider defini...
1,ncei_climate_at_a_glance_county_monthly,month,769440,1.0,12.0,0,review extreme min/max against provider defini...
2,ncei_climate_at_a_glance_county_monthly,value,769440,-17.1,112.2,980,review negative values/sentinels
3,ncei_climate_at_a_glance_county_monthly,anomaly,0,NaN,NaN,0,review extreme min/max against provider defini...
4,ncei_climate_at_a_glance_county_monthly,rank,0,NaN,NaN,0,review extreme min/max against provider defini...


## Source-specific limitations

County values are gridded/aggregated climate summaries, parameter coverage may differ by month, and ranks/anomalies depend on the provider reference period.

## Downstream readiness

**Assessment: PASS WITH LIMITATIONS when county-month-parameter keys are unique and the four required parameters (tavg, tmin, tmax, pcp) have adequate coverage.**

This assessment is conditional on the displayed inventories and checks. The normalized
`mart.*` builders—not this notebook—own parsing, suppression handling, geographic
resolution, deduplication, and downstream transformations.

In [8]:
con.close()